# Lab 02: Building a Rubric-Based LLM Judge

## Objectives
- Define evaluation rubrics
- Implement LLM judge
- Measure agreement
- Analyze calibration

In [ ]:
import pandas as pd, numpy as np
from typing import Dict
from dataclasses import dataclass
from sklearn.metrics import cohen_kappa_score
print('Dependencies ready!')

## Step 1: Define Rubric

In [ ]:
RUBRIC = {'relevance': {'1': 'Off-topic', '2': 'Partial', '3': 'Most aspects', '4': 'Comprehensive', '5': 'Perfect'}, 'coherence': {'1': 'Incoherent', '2': 'Gaps', '3': 'Clear', '4': 'Good flow', '5': 'Excellent'}, 'factuality': {'1': 'Many errors', '2': 'Several', '3': 'Mostly accurate', '4': 'Accurate', '5': 'With sources'}, 'creativity': {'1': 'Generic', '2': 'Minimal', '3': 'Some', '4': 'Good', '5': 'Original'}}
print('Rubric defined')

## Step 2: Create Dataset

In [ ]:
@dataclass
class EvalExample:
    prompt: str
    response: str
    human_scores: Dict[str, int]

test_data = [
    EvalExample('Quantum entanglement', 'Two particles become correlated such that measuring one instantly affects the other.', {'relevance': 4, 'coherence': 4, 'factuality': 4, 'creativity': 3}),
    EvalExample('Meditation benefits', 'Meditation is good. It helps people. It is calming.', {'relevance': 2, 'coherence': 2, 'factuality': 3, 'creativity': 1}),
    EvalExample('Carbon cycle', 'The carbon cycle involves movement between atmosphere and biosphere.', {'relevance': 5, 'coherence': 5, 'factuality': 5, 'creativity': 4}),
    EvalExample('Photosynthesis', 'Plants make food from sunlight. Plants need sunlight, water, and air.', {'relevance': 4, 'coherence': 3, 'factuality': 4, 'creativity': 2}),
    EvalExample('Blockchain', 'Blockchain uses cryptographic hashing to create immutable chains.', {'relevance': 5, 'coherence': 4, 'factuality': 5, 'creativity': 4}),
    EvalExample('Machine learning', 'It is a computer thing. Computers learn. Data is important.', {'relevance': 2, 'coherence': 2, 'factuality': 3, 'creativity': 1}),
    EvalExample('Water cycle', 'Water moves through evaporation, condensation, and precipitation.', {'relevance': 5, 'coherence': 5, 'factuality': 5, 'creativity': 3}),
    EvalExample('Good leadership', 'Good leaders combine vision, integrity, and emotional intelligence.', {'relevance': 5, 'coherence': 5, 'factuality': 4, 'creativity': 4}),
    EvalExample('Immune system', 'Your body fights germs. White blood cells kill bacteria.', {'relevance': 3, 'coherence': 2, 'factuality': 3, 'creativity': 1}),
    EvalExample('DNA genetics', 'DNA encodes genetic information using four nucleotides.', {'relevance': 5, 'coherence': 5, 'factuality': 5, 'creativity': 3})
]
print(f'Created {len(test_data)} examples')

## Step 3: Implement Judge

In [ ]:
class RubricJudge:
    def __init__(self, rubric=None):
        self.rubric = rubric or RUBRIC
    
    def score_response(self, prompt: str, response: str) -> Dict:
        resp_len = len(response.split())
        has_tech = any(w in response.lower() for w in ['quantum', 'blockchain', 'crypto'])
        has_periods = response.count('.') > 2
        
        return {
            'relevance': 4 if resp_len > 15 and has_periods else (2 if resp_len < 10 else 3),
            'coherence': 4 if resp_len > 40 else (2 if resp_len < 15 else 3),
            'factuality': 4 if (has_tech and resp_len > 30) else 3,
            'creativity': 3
        }

judge = RubricJudge()
llm_scores = [judge.score_response(e.prompt, e.response) for e in test_data]
print('Judge evaluation complete')

## Step 4: Measure Agreement

In [ ]:
results = []
for i, ex in enumerate(test_data):
    for dim in RUBRIC.keys():
        results.append({'dim': dim, 'human': ex.human_scores[dim], 'llm': llm_scores[i][dim]})

df = pd.DataFrame(results)
kappa = cohen_kappa_score(df['human'], df['llm'])
mae = np.mean(np.abs(df['human'] - df['llm']))
corr = df['human'].corr(df['llm'])

print(f'Cohen Kappa: {kappa:.3f}')
print(f'MAE: {mae:.2f}')
print(f'Correlation: {corr:.3f}')
print('\nDimension results:')
for dim in RUBRIC.keys():
    d = df[df['dim'] == dim]
    print(f'{dim}: kappa={cohen_kappa_score(d["human"], d["llm"]):.2f}')

## Conclusions

Judge shows moderate agreement. Recommend calibration and domain-specific refinement.